# Model inspection with treescope

Interactive view of `CNNBiLSTM`: module tree, weights, and the activations of one real PPG window. Click any node or array to expand it; hover arrays for values.

Run `viz/export_onnx.py` first only if you also want the Netron/Zetane view; this notebook needs just a checkpoint and `cache/windows/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
import treescope

ROOT = Path.cwd().resolve().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

from bpm.data_loading import load_subject_npz
from bpm.dataset import zscore_window
from evaluate import load_model

# shortcut: treescope 0.1.10 reads Tensor.names (named tensors, removed in torch 2.14); drop once treescope stops using it
if not hasattr(torch.Tensor, "names"):
    torch.Tensor.names = property(lambda t: (None,) * t.dim())

treescope.basic_interactive_setup(autovisualize_arrays=True)

CKPT = ROOT / "cache" / "best_huber10_fold0_seed42.pt"
SUBJECT = "multisite_P1"

## Module tree and weights

BatchNorm is shown unfused here (`running_mean`, `running_var`, affine weights), unlike the ONNX graph, where it is folded into the Conv.

In [2]:
model = load_model(CKPT).cpu()
model

CNNBiLSTM(
  (cnn): Sequential(
    (0): Conv1d(1, 32, kernel_size=(5,), stride=(1,), padding=same)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=same)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=same)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(128, 64, batch_first=True, bidirectional=True)
  (head): Sequential(
    (0): Linear(in_features=128, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

## Activations of one window

Forward hooks record every leaf module's output for one z-scored 10 s window (1000 samples at 100 Hz).

In [3]:
data = load_subject_npz(SUBJECT, ROOT / "cache" / "windows")
x = torch.from_numpy(zscore_window(data["windows"][:1]).astype(np.float32))

acts = {}
hooks = [
    m.register_forward_hook(lambda m, i, o, n=name: acts.__setitem__(n, o[0] if isinstance(o, tuple) else o))
    for name, m in model.named_modules() if not list(m.children())
]
with torch.no_grad():
    pred = model(x)
for h in hooks:
    h.remove()

print(f"label {data['labels'][0]:.1f} BPM, prediction {pred.flatten()[0]:.1f}")
acts

label 82.1 BPM, prediction 80.0


{'cnn.0': tensor([[[ 0.2477,  0.5024,  1.7589,  ...,  0.2713,  0.2596,  0.2902],
          [ 0.5028, -0.2876, -0.3471,  ...,  0.0547,  0.0415,  0.0744],
          [ 1.1747,  1.0157,  1.5859,  ..., -0.0307,  0.0132,  0.0399],
          ...,
          [ 1.2874,  2.1615,  2.3831,  ..., -0.1460, -0.1309, -0.0384],
          [-0.1347, -0.5057, -0.0452,  ..., -0.2238, -0.2050, -0.2433],
          [-0.1907,  0.3213, -0.5972,  ..., -0.1905, -0.1953, -0.1713]]]),
 'cnn.1': tensor([[[-9.7042e-02,  2.1874e-01,  1.7762e+00,  ..., -6.7738e-02,
           -8.2294e-02, -4.4356e-02],
          [ 2.0769e+00, -1.3194e+00, -1.5749e+00,  ...,  1.5123e-01,
            9.4454e-02,  2.3603e-01],
          [ 1.1593e+00,  1.0006e+00,  1.5700e+00,  ..., -4.4502e-02,
           -6.9650e-04,  2.6009e-02],
          ...,
          [ 3.0798e-01,  7.7560e-01,  8.9411e-01,  ..., -4.5890e-01,
           -4.5080e-01, -4.0132e-01],
          [ 9.4438e-01, -1.8182e+00,  1.6112e+00,  ...,  2.8092e-01,
            4.2114e-01,  1.3578e-01],
          [ 2.2185e-01,  2.8404e+00, -1.8567e+00,  ...,  2.2285e-01,
            1.9838e-01,  3.2128e-01]]]),
 'cnn.2': tensor([[[0.0000, 0.2187, 1.7762,  ..., 0.0000, 0.0000, 0.0000],
          [2.0769, 0.0000, 0.0000,  ..., 0.1512, 0.0945, 0.2360],
          [1.1593, 1.0006, 1.5700,  ..., 0.0000, 0.0000, 0.0260],
          ...,
          [0.3080, 0.7756, 0.8941,  ..., 0.0000, 0.0000, 0.0000],
          [0.9444, 0.0000, 1.6112,  ..., 0.2809, 0.4211, 0.1358],
          [0.2219, 2.8404, 0.0000,  ..., 0.2228, 0.1984, 0.3213]]]),
 'cnn.3': tensor([[[0.2187, 1.7762, 1.3360,  ..., 0.1365, 0.0000, 0.0000],
          [2.0769, 0.0000, 0.0000,  ..., 0.0933, 0.1512, 0.2360],
          [1.1593, 1.5700, 1.1137,  ..., 0.0000, 0.0000, 0.0260],
          ...,
          [0.7756, 0.8941, 0.5762,  ..., 0.0000, 0.0000, 0.0000],
          [0.9444, 1.6112, 0.9524,  ..., 0.1830, 0.2809, 0.4211],
          [2.8404, 0.0000, 0.0000,  ..., 0.1340, 0.2228, 0.3213]]]),
 'cnn.4': tensor([[[ 1.2772e+00,  5.8216e-01,  4.0907e-01,  ...,  7.1656e-01,
            9.5517e-01,  1.1078e+00],
          [-2.8353e-02,  4.9438e-01, -1.0231e+00,  ..., -1.5918e+00,
           -1.4918e+00, -9.8770e-01],
          [-5.4473e+00, -5.1487e+00, -8.8896e+00,  ..., -1.5913e+00,
           -1.2004e+00, -1.2571e+00],
          ...,
          [-1.0685e+00, -4.3053e+00, -7.4683e+00,  ..., -2.6840e+00,
           -1.7576e+00, -9.6752e-01],
          [-3.2199e+00, -2.8616e+00, -2.5577e+00,  ...,  6.7449e-01,
           -4.7370e-02, -7.9960e-01],
          [-3.7159e+00, -4.2583e+00, -5.4301e+00,  ...,  7.6598e-02,
            2.1361e-01, -8.7088e-03]]]),
 'cnn.5': tensor([[[-0.7609, -1.0176, -1.0815,  ..., -0.9679, -0.8798, -0.8234],
          [ 0.5524,  0.7285,  0.2174,  ...,  0.0258,  0.0595,  0.2293],
          [-0.5284, -0.4567, -1.3555,  ...,  0.3980,  0.4920,  0.4783],
          ...,
          [ 0.3096, -0.1657, -0.6302,  ...,  0.0723,  0.2084,  0.3244],
          [-1.0500, -0.9199, -0.8095,  ...,  0.3644,  0.1022, -0.1710],
          [-0.4305, -0.6096, -0.9965,  ...,  0.8217,  0.8669,  0.7935]]]),
 'cnn.6': tensor([[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.5524, 0.7285, 0.2174,  ..., 0.0258, 0.0595, 0.2293],
          [0.0000, 0.0000, 0.0000,  ..., 0.3980, 0.4920, 0.4783],
          ...,
          [0.3096, 0.0000, 0.0000,  ..., 0.0723, 0.2084, 0.3244],
          [0.0000, 0.0000, 0.0000,  ..., 0.3644, 0.1022, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.8217, 0.8669, 0.7935]]]),
 'cnn.7': tensor([[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.7285, 0.2174, 0.0000,  ..., 0.4059, 0.0258, 0.2293],
          [0.0000, 1.3909, 1.4331,  ..., 0.6809, 0.6864, 0.4920],
          ...,
          [0.3096, 0.0000, 0.0000,  ..., 0.0959, 0.0723, 0.3244],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.3644, 0.1022],
          [0.0000, 2.2492, 2.2804,  ..., 0.8660, 1.0475, 0.8669]]]),
 'cnn.8': tensor([[[  7.9298,   5.1190,  -2.8538,  .

## Input vs. conv feature maps

Channels on the first axis, time on the second. MaxPool halves the time axis each block (1000, 500, 250, 125).

In [4]:
treescope.render_array(x[0], axis_labels={0: "channel", 1: "time"})

<Arrayviz rendering>

In [5]:
treescope.render_array(acts["cnn.11"][0], axis_labels={0: "channel", 1: "time"})

<Arrayviz rendering>

## LSTM gates

PyTorch stacks the four gate matrices as rows of `weight_ih_l0`: input `i`, forget `f`, cell `g`, output `o`.

$i_t=\\sigma(W_{ii}x_t+W_{hi}h_{t-1}+b_i)$, $f_t=\\sigma(W_{if}x_t+W_{hf}h_{t-1}+b_f)$, $g_t=\\tanh(W_{ig}x_t+W_{hg}h_{t-1}+b_g)$, $o_t=\\sigma(W_{io}x_t+W_{ho}h_{t-1}+b_o)$

$c_t=f_t\\odot c_{t-1}+i_t\\odot g_t$, $h_t=o_t\\odot\\tanh(c_t)$

In [6]:
H = model.lstm.hidden_size
W = model.lstm.weight_ih_l0.detach().reshape(4, H, -1)  # (gate, hidden, cnn feature)
treescope.render_array(W, axis_labels={0: "gate i,f,g,o", 1: "hidden", 2: "cnn feature"})

<Arrayviz rendering>

In [7]:
treescope.render_array(acts["lstm"][0], axis_labels={0: "time", 1: "fwd|bwd hidden"})

<Arrayviz rendering>